# 03 — Gold Layer: Historical and Incremental Loads

This notebook performs a full Gold refresh in historical mode and a `detalle_id` MERGE for the current month and the two previous months in incremental mode.


In [ ]:
dbutils.widgets.text("tipo_carga", "Incremental")
dbutils.widgets.text("fecha_carga", "2025-06-30")

load_type = dbutils.widgets.get("tipo_carga").strip()
load_date = dbutils.widgets.get("fecha_carga").strip()

if load_type.lower() not in {"historico", "incremental"}:
    raise ValueError("tipo_carga must be 'Historico' or 'Incremental'.")


In [ ]:
CATALOG = "sales_store"
SCHEMA = "linio"

SILVER_PURCHASES_TABLE = f"{CATALOG}.{SCHEMA}.silver_compras"
SILVER_DETAILS_TABLE = f"{CATALOG}.{SCHEMA}.silver_detalles"
GOLD_FACT_TABLE = f"{CATALOG}.{SCHEMA}.gold_fact_compras"


In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp


## Define processing partitions


In [ ]:
reference_date = datetime.strptime(load_date, "%Y-%m-%d")

partition_dates = [
    (reference_date - relativedelta(months=2)).replace(day=1),
    (reference_date - relativedelta(months=1)).replace(day=1),
    reference_date.replace(day=1)
]

partition_values = [value.strftime("%Y-%m-%d") for value in partition_dates]
print("Partitions to process:", partition_values)


## Build the Gold fact DataFrame


In [ ]:
is_historical = load_type.lower() == "historico"

df_details = spark.table(SILVER_DETAILS_TABLE)

if is_historical:
    df_purchases = spark.table(SILVER_PURCHASES_TABLE)
else:
    df_purchases = (
        spark.table(SILVER_PURCHASES_TABLE)
        .filter(col("periodo").isin(partition_values))
    )

df_gold_source = (
    df_purchases.alias("purchases")
    .join(df_details.alias("details"), "factura", "inner")
    .select(
        col("purchases.periodo").alias("periodo"),
        col("purchases.venta_id").alias("venta_id"),
        col("purchases.factura").alias("factura"),
        col("purchases.tipo_compra").alias("tipo_compra"),
        col("purchases.fecha_orden").alias("fecha_orden"),
        col("purchases.fecha_entrega").alias("fecha_entrega"),
        col("purchases.fecha_envio").alias("fecha_envio"),
        col("purchases.estado").alias("estado"),
        col("purchases.cliente_id").alias("cliente_id"),
        col("purchases.vendedor").alias("vendedor"),
        col("purchases.departamento").alias("departamento"),
        col("purchases.metodo_pago").alias("metodo_pago"),
        col("purchases.grupo_dias_envio").alias("grupo_dias_envio"),
        col("details.detalle_id").alias("detalle_id"),
        col("details.producto_id").alias("producto_id"),
        col("details.unidades").alias("unidades"),
        col("details.subtotal").alias("subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )
)


## Load Gold


In [ ]:
if is_historical:
    (
        df_gold_source.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .partitionBy("periodo")
        .saveAsTable(GOLD_FACT_TABLE)
    )
else:
    (
        DeltaTable.forName(spark, GOLD_FACT_TABLE)
        .alias("target")
        .merge(
            df_gold_source.alias("source"),
            "target.detalle_id = source.detalle_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Gold load completed.")


## Validation


In [ ]:
display(
    spark.sql(f'''
        SELECT
            periodo,
            COUNT(DISTINCT factura) AS total_invoices,
            ROUND(SUM(subtotal), 2) AS total_sales
        FROM {GOLD_FACT_TABLE}
        GROUP BY periodo
        ORDER BY periodo
    ''')
)
